In [1]:
import numpy as np
import matplotlib.pyplot as plt
import random
import os
import shutil
import xarray as xr

In [ ]:
# Import new profile to APCEMM met file
# Open the NetCDF file for writing

# Create new met file to edit with RH data
met_name = "mu" + str(f"{cases[i].mu:.2f}") + "sigma" + str(f"{cases[i].sigma:.2f}")

dest_dir = '/home/chinahg/GCresearch/contrailuncertainty/APCEMM_results'
src_file = '/home/chinahg/GCresearch/APCEMM/examples/Example3_met_input/example_met_file.nc'
shutil.copy(src_file,dest_dir) #copy the file to destination dir

dst_file = os.path.join(dest_dir,'example_met_file.nc')
new_dst_file_name = os.path.join(dest_dir, met_name + 'met.nc')
# os.rename(dst_file, new_dst_file_name) #rename

# Edit nc file with RH profiles
nc_file = xr.open_dataset(src_file)

# Modify its values
#print(nc_file['relative_humidity'][:])
nc_file['relative_humidity'][:] = cases[i].RH
#print("New values for case " + str(i))
#print(nc_file['relative_humidity'][:])
    
# Close the NetCDF file to save changes
nc_file.to_netcdf(new_dst_file_name)
nc_file.close()

# Modify input.yaml file to take new meteorological netCDF file (changes saved but overwritten later)
import yaml

# Load the YAML file
with open("/home/chinahg/GCresearch/APCEMM/examples/Example3_met_input/input.yaml", "r") as file:
        data = yaml.safe_load(file)

# Modify the desired line
data["METEOROLOGY MENU"]["METEOROLOGICAL INPUT SUBMENU"]["Met input file path (string)"] = new_dst_file_name

# Save the modified data back to the YAML file
with open("/home/chinahg/GCresearch/APCEMM/examples/Example3_met_input/input.yaml", "w") as file:
    yaml.dump(data, file, default_flow_style=False)

# Then, run APCEMM with modified netCDF input file (see example 3)
!sbatch /home/chinahg/GCresearch/APCEMM/rundirs/SampleRunDir/uncertainty.sh